Task 1 – Large Scale Social Network Analysis 
Build a complete pipeline for identifying influential users in a large-scale social network. 
Requirements: 
• Extract a real-world dataset yourself (Reddit, Twitter/X, Mastodon, Telegram, GitHub, Wikipedia, or another 
publicly available network). If scraping is required, obtain the data ethically and document the collection process. 
• Before implementation, submit a one-page proposal describing: - the dataset you selected, - why it is suitable, - expected challenges, - preprocessing strategy, - expected graph size. 
• Clean and preprocess the dataset. 
• Construct the graph and justify whether it should be directed/undirected and weighted/unweighted. 
• Compare Degree, Betweenness, Eigenvector Centrality and PageRank. 
• Detect communities and analyse whether influential users remain stable across communities. 
• Discuss scalability, computational complexity and limitations if the graph size increases by 100×. 
• Explain why your final approach is preferable to alternative graph algorithms.

In [21]:
import gzip

with gzip.open("wiki-Vote.txt.gz", "rt") as file:
    
    for _ in range(10):
        line = file.readline()
        if not line:
            break 
        print(line, end="")

# Directed graph (each unordered pair of nodes is saved once): Wiki-Vote.txt 
# Wikipedia voting on promotion to administratorship (till January 2008). Directed edge A->B means user A voted on B becoming Wikipedia administrator.
# Nodes: 7115 Edges: 103689
# FromNodeId	ToNodeId
30	1412
30	3352
30	5254
30	5543
30	7478
3	28


In [8]:
import urllib.request
import pandas as pd

print("Extracting dataset...")
url = "https://snap.stanford.edu/data/wiki-Vote.txt.gz"
Our_File = "wiki-Vote.txt.gz"

# Programmatically downloading the dataset
urllib.request.urlretrieve(url, Our_File)


Clean_CSV = pd.read_csv(Our_File, sep='\t', comment='#', names=['source', 'target'], compression='gzip')

print(Clean_CSV.head())
t= Clean_CSV.shape
print("Data Shape :" ,t)


Extracting dataset...
   source  target
0      30    1412
1      30    3352
2      30    5254
3      30    5543
4      30    7478
Data Shape : (103689, 2)


In [14]:
import networkx as nx 
import pandas as pd
Graph= nx.from_pandas_edgelist(Clean_CSV, source='source', target='target', create_using= nx.DiGraph())
Self_Loop_Edges= list(nx.selfloop_edges(Graph))

Graph.remove_edges_from(Self_Loop_Edges)

largest_CC = max(nx.weakly_connected_components(Graph), key=len)

Graph_Core= Graph.subgraph(largest_CC).copy()

print(f"Graph constructed: {Graph_Core.number_of_nodes()} nodes, {Graph_Core.number_of_edges()} edges.")

Graph constructed: 7066 nodes, 103663 edges.


In [17]:
import time 

print("Calculating Centralities (this may take a minute)...")
start_time = time.time()

degree_cent = nx.in_degree_centrality(Graph_Core)

pagerank_cent = nx.pagerank(Graph_Core, alpha=0.85)      # Damping Factor 

eigen_cent = nx.eigenvector_centrality(Graph_Core, max_iter=1000) 

betweenness_cent = nx.betweenness_centrality(Graph_Core, seed=42)   # when k= 100 , it takes 4 sec, K=1000 it took 30 sec 

print(f"Metrics calculated in {round(time.time() - start_time, 2)} seconds.")

Cent_Metrices= pd.DataFrame({'In-Degree': degree_cent, 'Betweenness': betweenness_cent, 'Eigenvector': eigen_cent, 'PageRank': pagerank_cent})

print(Cent_Metrices.head())


Calculating Centralities (this may take a minute)...
Metrics calculated in 233.66 seconds.
      In-Degree  Betweenness  Eigenvector  PageRank
30     0.003255     0.000060     0.002351  0.000174
1412   0.004105     0.000000     0.001490  0.000817
3352   0.037367     0.004393     0.086794  0.001792
5254   0.037509     0.001550     0.078512  0.002158
5543   0.020524     0.002232     0.046956  0.001055


In [25]:
Cent_Metrics_Sorted = Cent_Metrices.sort_values(by= 'PageRank', ascending=False)

print("\nTop 5 rows of sorted metrics:")
print( Cent_Metrics_Sorted.head())
top10_influ = Cent_Metrics_Sorted.head(10).index.tolist()

print("The Top 10 PageRank Influencers (Node IDs): ",top10_influ)


Undir_Graph= Graph_Core.to_undirected()
communities = nx.community.louvain_communities(Undir_Graph, seed=42)

print(f"\nLouvain found {len(communities)} distinct communities.")

# 3. Write a loop to find which community each top influencer belongs to
print("\n--- Influencer Community Stability ---")

# communities is a list of sets. Each set contains the nodes for that community.
for influencer in top10_influ:
    # We use enumerate to get a community ID (index number)
    for community_id, node_set in enumerate(communities):
        if influencer in node_set:
            print(f"Influencer Node {influencer} belongs to Community {community_id}")
            break # Once found, stop searching and move to the next influencer


Top 5 rows of sorted metrics:
      In-Degree  Betweenness  Eigenvector  PageRank
4037   0.064685     0.003854     0.108955  0.004629
15     0.051097     0.011724     0.098179  0.003694
6634   0.028733     0.000758     0.053912  0.003538
2625   0.046851     0.000000     0.095526  0.003298
2398   0.048125     0.003681     0.117198  0.002615
The Top 10 PageRank Influencers (Node IDs):  [4037, 15, 6634, 2625, 2398, 2470, 2237, 4191, 7553, 5254]

Louvain found 5 distinct communities.

--- Influencer Community Stability ---
Influencer Node 4037 belongs to Community 1
Influencer Node 15 belongs to Community 3
Influencer Node 6634 belongs to Community 1
Influencer Node 2625 belongs to Community 0
Influencer Node 2398 belongs to Community 4
Influencer Node 2470 belongs to Community 0
Influencer Node 2237 belongs to Community 4
Influencer Node 4191 belongs to Community 3
Influencer Node 7553 belongs to Community 1
Influencer Node 5254 belongs to Community 3
